# Otto — Simulação, Modelagem e Validação com ROS 2, Gazebo, RViz, MATLAB e Octave

**Repositório:** https://github.com/inovamech/otto

Este material apresenta o fluxo completo utilizado no projeto Otto, desde a modelagem CAD no Fusion 360 até a validação cinemática e dinâmica em ROS 2, Gazebo, RViz, MATLAB e GNU Octave.

A proposta é documentar um processo de engenharia reproduzível, permitindo que o modelo desenvolvido em CAD seja convertido para Xacro, transformado em URDF, validado no ecossistema ROS 2 e posteriormente utilizado em ferramentas de análise e simulação.


---
## 1. Visão Geral do Projeto

O **Otto** é um robô bípede open-source amplamente utilizado em educação em robótica. Neste projeto ele é recriado digitalmente para validar:

- A **estrutura mecânica** (links, juntas, collisions)
- A **cinemática** antes de qualquer montagem física
- Os **algoritmos de controle** via ROS 2
- A **integração de sensores** (ultrassom, IMU) em ambiente simulado

### Fluxo Completo

| Etapa | Ferramenta | Saída |
|-------|-----------|-------|
| Modelagem 3D | Fusion 360 | `.f3d`, `.stl` |
| Descrição cinemática | URDF/Xacro | `.urdf`, `.xacro` |
| Validação visual | RViz2 | Visualização interativa |
| Simulação física | Gazebo Harmonic | Física + sensores |
| Controle | ROS 2 Jazzy | Nós Python |
| Infraestrutura | Docker | Reprodutibilidade |

### Estrutura de Diretórios

```
├── LICENSE
├── README.md
├── comandos.txt
├── docker
│   ├── Dockerfile
│   ├── docker-compose.yml
│   ├── entrypoint.sh
│   └── stl_report.txt
├── estrutura.txt
├── matlab
│   ├── COM-Projecao.png
│   ├── matlab.mat
│   └── matlabRev1.mat
├── octave
└── ros2_ws
    ├── build
    │   ├── COLCON_IGNORE
    │   └── otto
    ├── install
    ├── log
    ├── media
    │   └── OttoDeslizDeitado_Gravando 2026-06-08 094002.mp4
    ├── scene.txt
    └── src
        ├── config.yaml
        └── otto_description
```

## Modelagem do Otto no Fusion 360

### Estrutura correta do CAD

A qualidade do modelo URDF depende diretamente da forma como o robô é estruturado no Fusion 360.

Boas práticas adotadas:

- um componente por elo;
- origem posicionada no eixo do servo;
- juntas definidas no Fusion 360;
- evitar corpos soltos;
- evitar dependências geométricas entre elos distintos.

Problemas observados durante o desenvolvimento incluíram uso excessivo de Rigid Groups, componentes compartilhando referências geométricas e dependências entre pernas esquerda e direita, resultando em árvores cinemáticas incorretas e exportações inadequadas.


## Geração do Xacro com Fusion360Descriptor

O modelo CAD é convertido para uma descrição robótica por meio da extensão Fusion360Descriptor.

Fluxo adotado:

```text
Fusion 360
     ↓
Fusion360Descriptor
     ↓
otto.xacro
     ↓
otto.urdf
     ↓
ROS 2 / Gazebo / RViz / MATLAB
```

### Limitações observadas

- componentes rígidos podem ser exportados incorretamente;
- dependências entre componentes podem gerar juntas inválidas;
- referências cruzadas dificultam a geração correta da árvore cinemática.

Esses problemas ocorreram durante o desenvolvimento e exigiram reorganização da estrutura CAD antes da exportação.


---
## 2. Ambiente Docker

Toda a simulação roda dentro de um container Docker para garantir reprodutibilidade — não é necessário instalar ROS 2 ou Gazebo diretamente na máquina host.

### Pré-requisitos

- Docker Desktop instalado
- Windows 11 com WSL2 (recomendado) **ou** Linux nativo
- Windows 10: instalar [VcXsrv](https://sourceforge.net/projects/vcxsrv/) para suporte gráfico

### `Dockerfile`

```dockerfile
FROM osrf/ros:jazzy-desktop

# Dependências do sistema
RUN apt-get update && apt-get install -y \
    ros-jazzy-gz-ros2-control \
    ros-jazzy-ros2-control \
    ros-jazzy-ros2-controllers \
    ros-jazzy-joint-state-publisher-gui \
    ros-jazzy-xacro \
    python3-colcon-common-extensions \
    && rm -rf /var/lib/apt/lists/*

# Workspace
WORKDIR /ros2_ws
COPY ros2_ws/src ./src

# Build
RUN . /opt/ros/jazzy/setup.sh && \
    colcon build --symlink-install

COPY docker/entrypoint.sh /entrypoint.sh
RUN chmod +x /entrypoint.sh
ENTRYPOINT ["/entrypoint.sh"]
```

### `docker-compose.yml`

```yaml
services:
  otto:
    build: .
    container_name: otto
    environment:
      - DISPLAY=${DISPLAY}
      - WAYLAND_DISPLAY=${WAYLAND_DISPLAY}
      - XDG_RUNTIME_DIR=/tmp
    volumes:
      - /tmp/.X11-unix:/tmp/.X11-unix:rw
      - ./ros2_ws:/ros2_ws
    network_mode: host
    privileged: true
    stdin_open: true
    tty: true
```

### `entrypoint.sh`

```bash
#!/bin/bash
source /opt/ros/jazzy/setup.bash
source /ros2_ws/install/setup.bash
exec "$@"
```

### Comandos para Subir o Ambiente

```bash
# 1. Clonar o repositório
git clone https://github.com/inovamech/otto.git
cd otto

# 2. Construir e iniciar o container
cd docker
docker compose up -d

# 3. Entrar no container
docker exec -it otto bash

# 4. (Dentro do container) Build do workspace
cd /ros2_ws
colcon build --symlink-install
source install/setup.bash
```

> **Dica:** Use `docker exec -it otto bash` em abas separadas do terminal para abrir múltiplos terminais dentro do mesmo container.

---
## 3. Estrutura do Pacote ROS 2

O pacote principal chama-se `otto_description`. É um pacote do tipo **ament_python**[^1].
 
[^1]: Sistema de empacotamento do ROS 2 para pacotes desenvolvidos em Python, permitindo instalação e execução de nós Python via `ros2 run`..

### `package.xml`

```xml
<?xml version="1.0"?>
<package format="3">
  <name>otto_description</name>
  <version>0.1.0</version>
  <description>Descrição URDF e launches do robô Otto</description>
  <maintainer email="inovamech@exemplo.com">inovaMech</maintainer>
  <license>MIT</license>

  <buildtool_depend>ament_cmake</buildtool_depend>

  <exec_depend>robot_state_publisher</exec_depend>
  <exec_depend>joint_state_publisher_gui</exec_depend>
  <exec_depend>rviz2</exec_depend>
  <exec_depend>xacro</exec_depend>
  <exec_depend>ros2launch</exec_depend>
  <exec_depend>gz_ros2_control</exec_depend>
  <exec_depend>ros2_controllers</exec_depend>

  <export>
    <build_type>ament_cmake</build_type>
  </export>
</package>
```

### `CMakeLists.txt`

```cmake
cmake_minimum_required(VERSION 3.8)
project(otto_description)

find_package(ament_cmake REQUIRED)

install(DIRECTORY
  urdf meshes config launch
  DESTINATION share/${PROJECT_NAME}
)

ament_package()
```

### Estrutura Atual do Projeto

```text
.
├── LICENSE
├── README.md
├── comandos.txt
├── estrutura.txt
├── docker
│   ├── Dockerfile
│   ├── docker-compose.yml
│   ├── entrypoint.sh
│   └── stl_report.txt
├── matlab
│   ├── COM-Projecao.png
│   ├── matlab.mat
│   └── matlabRev1.mat
├── octave
└── ros2_ws
    ├── build
    ├── install
    ├── log
    ├── media
    ├── scene.txt
    └── src
        ├── config.yaml
        └── otto_description
```


---
## 4. URDF e Xacro — Descrição do Robô

O **URDF** (Unified Robot Description Format) descreve a geometria, inércia e juntas do Otto.  
Usamos **Xacro** para parametrizar e reutilizar trechos do código.

### Anatomia do Otto (Links e Joints)

```
base_link
 ├── left_hip_link       (revolute — eixo Y)
 │    └── left_knee_link (revolute — eixo Y)
 │         └── left_foot_link (fixed)
 └── right_hip_link      (revolute — eixo Y)
      └── right_knee_link (revolute — eixo Y)
           └── right_foot_link (fixed)
```

### `otto.urdf.xacro` — Estrutura Completa


In [ ]:

```xml
<?xml version="1.0"?>
<robot xmlns:xacro="http://www.ros.org/wiki/xacro" name="otto">

  <!-- ========== PROPRIEDADES ========== -->
  <xacro:property name="body_mass"  value="0.3" />
  <xacro:property name="leg_mass"   value="0.05" />
  <xacro:property name="body_x"     value="0.07" />
  <xacro:property name="body_y"     value="0.06" />
  <xacro:property name="body_z"     value="0.04" />

  <!-- ========== MACRO INÉRCIA BOX ========== -->
  <xacro:macro name="inertia_box" params="m x y z">
    <inertial>
      <mass value="${m}" />
      <inertia
        ixx="${m*(y*y+z*z)/12}" ixy="0" ixz="0"
        iyy="${m*(x*x+z*z)/12}" iyz="0"
        izz="${m*(x*x+y*y)/12}" />
    </inertial>
  </xacro:macro>

  <!-- ========== BASE ========== -->
  <link name="base_link">
    <visual>
      <geometry>
        <mesh filename="package://otto_description/meshes/base_link.stl" />
      </geometry>
      <material name="blue">
        <color rgba="0.0 0.5 1.0 1.0" />
      </material>
    </visual>
    <collision>
      <geometry>
        <box size="${body_x} ${body_y} ${body_z}" />
      </geometry>
    </collision>
    <xacro:inertia_box m="${body_mass}" x="${body_x}" y="${body_y}" z="${body_z}" />
  </link>

  <!-- ========== MACRO PERNA ========== -->
  <xacro:macro name="leg" params="side reflect">

    <link name="${side}_hip_link">
      <visual>
        <geometry>
          <mesh filename="package://otto_description/meshes/${side}_hip.stl" />
        </geometry>
      </visual>
      <collision>
        <geometry><box size="0.02 0.03 0.04" /></geometry>
      </collision>
      <xacro:inertia_box m="${leg_mass}" x="0.02" y="0.03" z="0.04" />
    </link>

    <joint name="${side}_hip_joint" type="revolute">
      <parent link="base_link" />
      <child  link="${side}_hip_link" />
      <origin xyz="${reflect * 0.025} 0 -0.02" rpy="0 0 0" />
      <axis xyz="0 1 0" />
      <limit lower="-0.523" upper="0.523" effort="1.5" velocity="3.14" />
    </joint>

    <link name="${side}_knee_link">
      <visual>
        <geometry>
          <mesh filename="package://otto_description/meshes/${side}_knee.stl" />
        </geometry>
      </visual>
      <collision>
        <geometry><box size="0.02 0.02 0.04" /></geometry>
      </collision>
      <xacro:inertia_box m="${leg_mass}" x="0.02" y="0.02" z="0.04" />
    </link>

    <joint name="${side}_knee_joint" type="revolute">
      <parent link="${side}_hip_link" />
      <child  link="${side}_knee_link" />
      <origin xyz="0 0 -0.04" rpy="0 0 0" />
      <axis xyz="0 1 0" />
      <limit lower="-0.523" upper="0.523" effort="1.5" velocity="3.14" />
    </joint>

    <link name="${side}_foot_link">
      <visual>
        <geometry>
          <mesh filename="package://otto_description/meshes/${side}_foot.stl" />
        </geometry>
      </visual>
      <collision>
        <geometry><box size="0.04 0.025 0.01" /></geometry>
      </collision>
      <xacro:inertia_box m="0.02" x="0.04" y="0.025" z="0.01" />
    </link>

    <joint name="${side}_foot_joint" type="fixed">
      <parent link="${side}_knee_link" />
      <child  link="${side}_foot_link" />
      <origin xyz="0 0 -0.04" rpy="0 0 0" />
    </joint>

  </xacro:macro>

  <!-- Instanciar pernas -->
  <xacro:leg side="left"  reflect="1" />
  <xacro:leg side="right" reflect="-1" />

  <!-- ========== SENSOR ULTRASSOM ========== -->
  <link name="ultrasound_link">
    <visual>
      <geometry><box size="0.02 0.04 0.01" /></geometry>
      <material name="gray"><color rgba="0.5 0.5 0.5 1.0" /></material>
    </visual>
  </link>
  <joint name="ultrasound_joint" type="fixed">
    <parent link="base_link" />
    <child  link="ultrasound_link" />
    <origin xyz="0.04 0 0.01" rpy="0 0 0" />
  </joint>

  <!-- ========== PLUGINS ROS2_CONTROL ========== -->
  <ros2_control name="otto_control" type="system">
    <hardware>
      <plugin>gz_ros2_control/GazeboSimSystem</plugin>
    </hardware>
    <joint name="left_hip_joint">
      <command_interface name="position" />
      <state_interface name="position" />
      <state_interface name="velocity" />
    </joint>
    <joint name="left_knee_joint">
      <command_interface name="position" />
      <state_interface name="position" />
      <state_interface name="velocity" />
    </joint>
    <joint name="right_hip_joint">
      <command_interface name="position" />
      <state_interface name="position" />
      <state_interface name="velocity" />
    </joint>
    <joint name="right_knee_joint">
      <command_interface name="position" />
      <state_interface name="position" />
      <state_interface name="velocity" />
    </joint>
  </ros2_control>

  <!-- Plugin Gazebo para ros2_control -->
  <gazebo>
    <plugin filename="gz_ros2_control-system"
            name="gz_ros2_control::GazeboSimROS2ControlPlugin">
      <parameters>$(find otto_description)/config/controllers.yaml</parameters>
    </plugin>
  </gazebo>

  <!-- Plugin sensor ultrassom -->
  <gazebo reference="ultrasound_link">
    <sensor name="ultrasound" type="gpu_ray">
      <pose>0 0 0 0 0 0</pose>
      <ray>
        <scan><horizontal><samples>1</samples><resolution>1</resolution>
          <min_angle>0</min_angle><max_angle>0</max_angle></horizontal></scan>
        <range><min>0.02</min><max>4.0</max><resolution>0.01</resolution></range>
      </ray>
      <plugin filename="gz-sim-ray-sensor-system"
              name="gz::sim::systems::RaySensor" />
      <always_on>true</always_on>
      <update_rate>10</update_rate>
    </sensor>
  </gazebo>

</robot>
```



### Processar o Xacro via Python

In [ ]:
import subprocess
import os

# Caminho do arquivo xacro
xacro_file = "/ros2_ws/src/otto_description/urdf/otto.urdf.xacro"
urdf_output = "/ros2_ws/src/otto_description/urdf/otto.urdf"

# Converter xacro -> urdf
result = subprocess.run(
    ["xacro", xacro_file, "-o", urdf_output],
    capture_output=True, text=True
)

if result.returncode == 0:
    print("✅ URDF gerado com sucesso em:", urdf_output)
    size = os.path.getsize(urdf_output)
    print(f"   Tamanho: {size} bytes")
else:
    print("❌ Erro ao gerar URDF:")
    print(result.stderr)

In [ ]:
# Validar URDF com check_urdf
result = subprocess.run(
    ["check_urdf", urdf_output],
    capture_output=True, text=True
)
print(result.stdout)
if result.stderr:
    print("Warnings:", result.stderr)

In [ ]:
# Gerar árvore cinemática em dot (GraphViz)
result = subprocess.run(
    ["urdf_to_graphviz", urdf_output],
    capture_output=True, text=True,
    cwd="/tmp"
)
print(result.stdout or "Arquivo .dot gerado em /tmp/otto.dot")

# Converter para PNG (requer graphviz)
subprocess.run(["dot", "-Tpng", "/tmp/otto.dot", "-o", "/tmp/otto_tree.png"])
print("Árvore salva em /tmp/otto_tree.png")

---
## 5. Visualização no RViz

O RViz2 é o visualizador 3D padrão do ROS 2. Ele permite inspecionar o modelo, a árvore de transforms (TF) e os dados de sensores em tempo real.

### `display.launch.py`

```python
import os
from ament_index_python.packages import get_package_share_directory
from launch import LaunchDescription
from launch.actions import DeclareLaunchArgument
from launch.substitutions import LaunchConfiguration, Command
from launch_ros.actions import Node
from launch_ros.parameter_descriptions import ParameterValue

def generate_launch_description():
    pkg = get_package_share_directory('otto_description')
    xacro_file = os.path.join(pkg, 'urdf', 'otto.urdf.xacro')
    rviz_config = os.path.join(pkg, 'config', 'otto.rviz')

    # Processar xacro em tempo de launch
    robot_description = ParameterValue(
        Command(['xacro ', xacro_file]),
        value_type=str
    )

    return LaunchDescription([
        # Publicar a descrição do robô
        Node(
            package='robot_state_publisher',
            executable='robot_state_publisher',
            parameters=[{'robot_description': robot_description}]
        ),
        # Slider para mover as juntas manualmente
        Node(
            package='joint_state_publisher_gui',
            executable='joint_state_publisher_gui',
        ),
        # Abrir o RViz com a configuração salva
        Node(
            package='rviz2',
            executable='rviz2',
            arguments=['-d', rviz_config],
        ),
    ])
```

### Executar

```bash
# Dentro do container
ros2 launch otto_description display.launch.py
```

### Configuração RViz (`otto.rviz`)

Displays importantes a habilitar no RViz:

| Display | Descrição |
|---------|----------|
| `RobotModel` | Visualiza links, meshes e collisions |
| `TF` | Exibe a árvore de frames de referência |
| `LaserScan` | Mostra dados do sensor ultrassom |
| `Axes` | Referencial global |

**Fixed Frame:** `base_link`

> ** Dica:** Com o `joint_state_publisher_gui` aberto, mova os sliders para animar as juntas do Otto em tempo real no RViz.

### O que verificar no RViz

- orientação dos eixos;
- posição dos pés;
- árvore TF;
- escala das malhas STL;
- inversões de junta.

### Problemas comuns

| Sintoma | Possível causa |
|----------|----------|
| Pé invertido | eixo da junta incorreto |
| Malha deslocada | origem do componente incorreta |
| Elo ausente | caminho STL inválido |


### Simulação no Gazebo Harmonic

O launch do projeto implementa um fluxo completo de visualização e simulação do robô Otto utilizando ROS 2 Jazzy e Gazebo Harmonic. Diferentemente de exemplos genéricos encontrados na documentação do ROS 2, o processo adotado neste projeto realiza a leitura direta do arquivo `otto.urdf`, publica sua descrição no ecossistema ROS 2, inicializa o RViz para inspeção visual e, em seguida, cria a entidade física correspondente dentro do Gazebo.

#### Fluxo de execução

```text
otto.urdf
     ↓
robot_description
     ↓
robot_state_publisher
     ↓
RViz
     ↓
ros_gz_sim (gz_sim.launch.py)
     ↓
Gazebo Harmonic
     ↓
create
(spawn da entidade otto)
     ↓
ros_gz_bridge
     ↓
/joint_states e /clock
```

O processo executado pelo `gazebo.launch.py` realiza:

1. leitura direta do arquivo `otto.urdf`;
2. publicação da descrição em `robot_description`;
3. execução do `robot_state_publisher`;
4. abertura simultânea do RViz;
5. inicialização do Gazebo Harmonic;
6. criação da entidade `otto`;
7. configuração da bridge ROS 2 ↔ Gazebo;
8. remapeamento dos estados das juntas para `/joint_states`.

#### Componentes executados

O launch inicia os seguintes componentes:

* `robot_state_publisher`
* `joint_state_publisher_gui`
* `rviz2`
* `ros_gz_sim`
* `ros_gz_bridge`

#### Inserção do robô no mundo virtual

O robô é inserido automaticamente no mundo `empty.sdf` utilizando:

```python
arguments=[
    '-topic', 'robot_description',
    '-name', 'otto',
    '-z', '0.033',
    '-Y', '3.1407',
]
```

Assim, a entidade é criada com altura inicial de aproximadamente 33 mm e orientação próxima de π radianos em torno do eixo vertical.

#### Comunicação entre ROS 2 e Gazebo

A sincronização entre o simulador e o ROS 2 é realizada pelo `ros_gz_bridge`.

```text
/world/empty/model/otto/joint_state
                     ↓
                 /joint_states

/clock
```

Dessa forma, os estados das juntas produzidos pelo Gazebo tornam-se acessíveis através do tópico ROS 2 `/joint_states`, enquanto o tempo de simulação é disponibilizado em `/clock`.

#### Observação sobre URDF e Xacro

Nesta versão do projeto, o launch utiliza diretamente o arquivo `otto.urdf`, previamente gerado a partir do processo de exportação do Fusion 360.

Não há processamento de arquivos Xacro durante a execução do launch. O fluxo adotado é:

```text
Fusion 360
     ↓
Fusion360Descriptor
     ↓
otto.xacro
     ↓
otto.urdf
     ↓
gazebo.launch.py
```

Portanto, toda a modelagem e parametrização ocorre antes da execução da simulação.

#### Verificações básicas

```bash
ros2 topic list

ros2 topic echo /robot_description

ros2 topic echo /joint_states

ros2 topic echo /clock

ros2 node list
```

#### Diagnósticos adicionais

```bash
ros2 topic hz /joint_states

ros2 run tf2_tools view_frames

ros2 control list_controllers

ros2 control list_hardware_interfaces
```

Essas verificações permitem confirmar a correta publicação da descrição do robô, a atualização dos estados das juntas, a sincronização temporal do simulador e o funcionamento da árvore TF utilizada pelo RViz e pelos algoritmos de controle.


---
## 7. Controle do Robô

O controle usa **ros2_control** com `JointTrajectoryController` para posicionamento das juntas.

### `controllers.yaml`

```yaml
controller_manager:
  ros__parameters:
    update_rate: 100  # Hz

    joint_state_broadcaster:
      type: joint_state_broadcaster/JointStateBroadcaster

    otto_leg_controller:
      type: joint_trajectory_controller/JointTrajectoryController

otto_leg_controller:
  ros__parameters:
    joints:
      - left_hip_joint
      - left_knee_joint
      - right_hip_joint
      - right_knee_joint
    command_interfaces:
      - position
    state_interfaces:
      - position
      - velocity
    state_publish_rate: 50.0
    action_monitor_rate: 20.0
    allow_partial_joints_goal: false
```

### `control.launch.py`

```python
from launch import LaunchDescription
from launch_ros.actions import Node

def generate_launch_description():
    return LaunchDescription([
        Node(
            package='controller_manager',
            executable='spawner',
            arguments=['joint_state_broadcaster', '--controller-manager', '/controller_manager'],
        ),
        Node(
            package='controller_manager',
            executable='spawner',
            arguments=['otto_leg_controller', '--controller-manager', '/controller_manager'],
        ),
    ])
```

### Executar

```bash
# Terminal 2 (após gazebo.launch.py estar rodando)
ros2 launch otto_description control.launch.py
```

### Nó de Controle de Marcha (`otto_walk.py`)

```python
#!/usr/bin/env python3
"""
otto_walk.py — Envia trajetórias senoidais para simular marcha bípede.
"""
import rclpy
from rclpy.node import Node
from trajectory_msgs.msg import JointTrajectory, JointTrajectoryPoint
from builtin_interfaces.msg import Duration
import math
import time

class OttoWalk(Node):
    def __init__(self):
        super().__init__('otto_walk')
        self.pub = self.create_publisher(
            JointTrajectory,
            '/otto_leg_controller/joint_trajectory',
            10
        )
        # Parâmetros de marcha
        self.amplitude = 0.3   # radianos
        self.period    = 1.0   # segundos por ciclo
        self.dt        = 0.05  # período de atualização

        self.timer = self.create_timer(self.dt, self.walk_callback)
        self.t = 0.0
        self.get_logger().info('OttoWalk iniciado!')

    def walk_callback(self):
        self.t += self.dt
        w = 2 * math.pi / self.period

        # Padrão senoidal com desfasamento de 90° entre os lados
        lh =  self.amplitude * math.sin(w * self.t)
        lk = -abs(self.amplitude * math.sin(w * self.t))  # joelho sempre flexiona
        rh = -self.amplitude * math.sin(w * self.t)       # fase oposta
        rk = -abs(self.amplitude * math.sin(w * self.t))

        msg = JointTrajectory()
        msg.joint_names = [
            'left_hip_joint', 'left_knee_joint',
            'right_hip_joint', 'right_knee_joint'
        ]
        pt = JointTrajectoryPoint()
        pt.positions = [lh, lk, rh, rk]
        pt.velocities = [0.0] * 4
        pt.time_from_start = Duration(sec=0, nanosec=int(self.dt * 1e9))
        msg.points = [pt]

        self.pub.publish(msg)

def main(args=None):
    rclpy.init(args=args)
    node = OttoWalk()
    rclpy.spin(node)
    node.destroy_node()
    rclpy.shutdown()

if __name__ == '__main__':
    main()
```

**Executar o nó:**

```bash
# Terminal 3
ros2 run otto_description otto_walk
```

---
## 8. Tópicos e Comunicação ROS 2

Os tópicos mais importantes para validação e depuração do sistema são:

```text
/robot_description
/joint_states
/clock
/tf
/tf_static
```

### Verificações básicas

```bash
ros2 topic list

ros2 topic echo /joint_states

ros2 topic echo /clock
```

### Comandos adicionais

```bash
ros2 topic hz /joint_states
ros2 run tf2_tools view_frames
ros2 control list_controllers
ros2 control list_hardware_interfaces
```


### Monitorar Tópicos via Python

In [ ]:
import subprocess
import json

def ros2_topic_list():
    """Lista tópicos ROS 2 ativos."""
    result = subprocess.run(
        ["ros2", "topic", "list"],
        capture_output=True, text=True
    )
    topics = result.stdout.strip().split('\n')
    return topics

def ros2_topic_info(topic):
    """Retorna informações de um tópico."""
    result = subprocess.run(
        ["ros2", "topic", "info", topic],
        capture_output=True, text=True
    )
    return result.stdout

# Executar dentro do container ROS
print("Tópicos ativos:")
for t in ros2_topic_list():
    print(f"  {t}")

---
## 9. Validação e Debug

### Checklist

```text
[ ] Docker iniciado

[ ] Workspace compilado

[ ] URDF validado

[ ] RViz carregado

[ ] Gazebo iniciado

[ ] joint_states recebidos

[ ] TF publicada

[ ] Controladores ativos
```

### Diagnósticos URDF


In [ ]:
import subprocess

urdf_file = "/ros2_ws/src/otto_description/urdf/otto.urdf"

# 1. Validar URDF
print("=== check_urdf ===")
r = subprocess.run(["check_urdf", urdf_file], capture_output=True, text=True)
print(r.stdout or r.stderr)

# 2. Contar links e juntas
with open(urdf_file) as f:
    content = f.read()

links  = content.count('<link ')
joints = content.count('<joint ')
print(f"\nLinks:  {links}")
print(f"Joints: {joints}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Simular trajetória de marcha no tempo
t = np.linspace(0, 4, 400)  # 4 segundos
T = 1.0        # período (s)
A = 0.3        # amplitude (rad)
w = 2 * np.pi / T

left_hip   =  A * np.sin(w * t)
right_hip  = -A * np.sin(w * t)
left_knee  = -np.abs(A * np.sin(w * t))
right_knee = -np.abs(A * np.sin(w * t))

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

axes[0].plot(t, np.degrees(left_hip),   label='Quadril Esq.', color='steelblue')
axes[0].plot(t, np.degrees(right_hip),  label='Quadril Dir.', color='tomato', linestyle='--')
axes[0].set_ylabel('Ângulo (graus)')
axes[0].set_title('Trajetória de Marcha Otto — Quadril')
axes[0].legend(); axes[0].grid(True)

axes[1].plot(t, np.degrees(left_knee),  label='Joelho Esq.',  color='steelblue')
axes[1].plot(t, np.degrees(right_knee), label='Joelho Dir.',  color='tomato', linestyle='--')
axes[1].set_xlabel('Tempo (s)')
axes[1].set_ylabel('Ângulo (graus)')
axes[1].set_title('Trajetória de Marcha Otto — Joelho')
axes[1].legend(); axes[1].grid(True)

plt.tight_layout()
plt.savefig('/tmp/otto_gait.png', dpi=150)
plt.show()
print("Gráfico salvo em /tmp/otto_gait.png")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def forward_kinematics_otto(hip_angle_deg, knee_angle_deg,
                            hip_length=0.04, knee_length=0.04):
    """
    Cinemática direta simplificada de uma perna do Otto (2 DOF, plano sagital).
    Retorna posição do pé (x, z) relativa ao quadril.
    """
    q1 = np.radians(hip_angle_deg)
    q2 = np.radians(knee_angle_deg)

    # Posição do joelho
    knee_x =  hip_length * np.sin(q1)
    knee_z = -hip_length * np.cos(q1)

    # Posição do pé
    foot_x = knee_x + knee_length * np.sin(q1 + q2)
    foot_z = knee_z - knee_length * np.cos(q1 + q2)

    return (knee_x, knee_z), (foot_x, foot_z)

# Plotar workspace da perna
fig, ax = plt.subplots(figsize=(7, 7))

for hip_a in np.linspace(-30, 30, 13):
    for knee_a in np.linspace(-30, 0, 7):
        (kx, kz), (fx, fz) = forward_kinematics_otto(hip_a, knee_a)
        ax.plot([0, kx, fx], [0, kz, fz], 'b-', alpha=0.15, linewidth=0.8)
        ax.plot(fx, fz, 'r.', markersize=3)

# Destacar postura ereta
(kx0, kz0), (fx0, fz0) = forward_kinematics_otto(0, 0)
ax.plot([0, kx0, fx0], [0, kz0, fz0], 'k-', linewidth=3, label='Postura ereta')
ax.plot(0, 0, 'gs', markersize=10, label='Quadril')
ax.plot(kx0, kz0, 'bs', markersize=8, label='Joelho')
ax.plot(fx0, fz0, 'rs', markersize=8, label='Pé')

ax.set_aspect('equal')
ax.grid(True)
ax.set_xlabel('X (m)')
ax.set_ylabel('Z (m) — positivo para cima')
ax.set_title('Workspace da Perna do Otto (Cinemática Direta)')
ax.legend()
plt.tight_layout()
plt.savefig('/tmp/otto_workspace.png', dpi=150)
plt.show()

### Problemas Comuns e Soluções

| Sintoma | Causa Provável | Solução |
|---------|---------------|--------|
| `check_urdf` falha com `joint without parent` | Junta sem `<parent>` no URDF | Verificar hierarquia no xacro |
| RViz abre mas robô não aparece | `robot_description` não publicado | Checar se `robot_state_publisher` está rodando |
| Gazebo crasha ao inserir robô | Colisão mal definida ou inércias zero | Revisar `<inertia>` de todos os links |
| Controlador não ativa | Plugin `gz_ros2_control` não carregado | Verificar URDF `<ros2_control>` e `<gazebo>` |
| Sem display gráfico | Variável `DISPLAY` não configurada | No Linux: `xhost +local:docker`; no WSL2: verificar VcXsrv |
| `colcon build` falha | Dependências faltando | `rosdep install --from-paths src --ignore-src -r -y` |

### Comandos de Debug

```bash
# Ver log do robot_state_publisher
ros2 run robot_state_publisher robot_state_publisher \
    --ros-args -p robot_description:="$(cat /ros2_ws/src/otto_description/urdf/otto.urdf)"

# Checar se controladores subiram
ros2 control list_controllers

# Publicar uma trajetória manualmente (teste rápido)
ros2 topic pub --once /otto_leg_controller/joint_trajectory \
    trajectory_msgs/msg/JointTrajectory \
    '{joint_names: [left_hip_joint, left_knee_joint, right_hip_joint, right_knee_joint],
      points: [{positions: [0.2, -0.1, -0.2, -0.1], time_from_start: {sec: 1}}]}'

# Inspecionar TF
ros2 run tf2_ros tf2_echo base_link left_foot_link
```

## MATLAB e Simulink

O arquivo de validação utilizado no projeto já realiza cálculos de massa total, centro de massa, transformações homogêneas e inspeção dos elos do robô.

### Carregamento do robô

```matlab
robot = importrobot("otto.urdf");
showdetails(robot)
```

### Massa total

```matlab
totalMass = 0;

for i = 1:numel(robot.Bodies)
    totalMass = totalMass + robot.Bodies{i}.Mass;
end

disp(totalMass)
```

### Centro de massa

```matlab
centerOfMass(robot);
```

### Propriedades inerciais

```matlab
body = robot.Bodies{1};

body.Mass
body.CenterOfMass
body.Inertia
```

### Transformações homogêneas

```matlab
T = getTransform(robot,...
                 config,...
                 "FootLeft_1");
```

Resultados previamente obtidos incluíram massa total aproximada de 0,818 kg e cálculo do centro de massa global do modelo.


## GNU Octave

O GNU Octave oferece uma alternativa livre para estudos de cinemática, planejamento de trajetória e análise matemática do robô.

Exemplo simplificado:

```octave
L1 = 0.05;
L2 = 0.06;

theta1 = deg2rad(20);
theta2 = deg2rad(-15);

x = L1*cos(theta1)+L2*cos(theta1+theta2);
z = L1*sin(theta1)+L2*sin(theta1+theta2);
```

Embora não possua integração nativa equivalente ao Robotics System Toolbox, é suficiente para diversas análises acadêmicas e atividades de ensino.


---
## 10. Referências

- ROS 2 Documentation
- Gazebo Harmonic Documentation
- URDF XML Specification
- Xacro Documentation
- Fusion360Descriptor
- MATLAB Robotics System Toolbox
- Simscape Multibody
- GNU Octave

## Considerações finais

Este trabalho apresentou um fluxo integrado para modelagem, validação e simulação do robô Otto utilizando Fusion 360, Fusion360Descriptor, ROS 2 Jazzy, RViz, Gazebo Harmonic, MATLAB e GNU Octave. A abordagem adotada permitiu estabelecer uma cadeia contínua de desenvolvimento, na qual um único modelo mecânico é utilizado desde a etapa de projeto CAD até as análises computacionais e simulações físicas, reduzindo inconsistências entre diferentes ferramentas e simplificando o processo de manutenção do sistema.

Os resultados obtidos demonstraram que a exportação do modelo desenvolvido no Fusion 360 para os formatos Xacro e URDF possibilita sua reutilização em múltiplos ambientes sem necessidade de reconstrução da estrutura do robô. No ROS 2 e no RViz foi possível validar a árvore cinemática, os sistemas de coordenadas e a consistência das transformações espaciais. No Gazebo Harmonic, o modelo foi inserido em um ambiente de simulação física integrado ao ROS 2 por meio do ros_gz_bridge, permitindo a troca de informações de estado e a avaliação do comportamento do robô em ambiente virtual. Complementarmente, a importação do URDF no MATLAB e no GNU Octave possibilitou a análise de propriedades estruturais, incluindo massa total, centro de massa, transformações homogêneas e parâmetros inerciais.

A principal contribuição do fluxo proposto é a utilização de uma única fonte de verdade para a descrição do robô. Alterações realizadas no modelo CAD podem ser propagadas de forma sistemática para os ambientes de visualização, simulação e análise, aumentando a rastreabilidade das modificações e reduzindo retrabalho durante o desenvolvimento. Além disso, a integração entre ferramentas amplamente utilizadas na pesquisa e na indústria aproxima o ambiente educacional de práticas profissionais empregadas no desenvolvimento de sistemas robóticos.

Como trabalhos futuros, pretende-se incorporar ROS 2 Control, modelos dinâmicos mais detalhados, sensores virtuais e algoritmos de controle, percepção e navegação. Também estão previstas investigações envolvendo planejamento de movimento, integração com técnicas de inteligência artificial e validação de estratégias de controle autônomo. Em uma etapa posterior, o modelo poderá servir como base para a implementação de um gêmeo digital do robô Otto, permitindo a sincronização entre os ambientes virtual e físico e ampliando as possibilidades de pesquisa, ensino e desenvolvimento em robótica educacional